# Phase 5 — ColQwen2.5 Visual Retrieval (Colab Pro L4)

**This notebook produces the REAL VISUAL-01 / VISUAL-02 retrieval-quality numbers.**
It is the phase reproducibility deliverable and the *no-fabrication boundary*: every
retrieval-quality number reported for this phase comes from running this notebook end-to-end
on a real GPU (Colab Pro **L4**, bf16). No offline test asserts any of these numbers.

Pipeline: install pinned stack -> load `vidore/colqwen2.5-v0.2` -> embed each corpus page
**from `pages.image_blob`** (NOT the gitignored source PDFs) -> build the three-named-vector
`sdf_page_images` Qdrant collection -> two-stage retrieval (mean-pooled HNSW prefetch ->
full-multivector MAX_SIM rerank) -> evaluate text-only vs visual-fused on the gold set ->
print the metrics table -> prove the four `rq_ex3_*` image-only gold pages appear in visual top-k.

The notebook imports the SAME pure builders the offline suite covers
(`src.retrieval.visual.{collection,pooling,querier,fusion,run}` and the
`src.retrieval.visual.embedder` lazy seam), so the plumbing is identical to what is unit-tested;
only the GPU forward pass and the printed numbers are notebook-exclusive.

**Setup:** Runtime -> Change runtime type -> **L4 GPU**, then upload `compliance.db` (with
`pages.image_blob`) when prompted, then Run All. `compliance.db` is NEVER committed.
No API keys / secrets are needed (retrieval is provider-free).

## 1. Install the pinned stack

Pins are **colpali-engine 0.3.17's actual requirements**: `transformers>=5.3,<6`,
`torch>=2.2,<2.12`, `peft>=0.18,<0.20`. (The earlier `transformers>=4.45,<4.50` pin was stale —
colpali 0.3.17 moved to the transformers-5 line, so the old pin fails the pip resolver.)
This install swaps Colab's preinstalled torch/transformers, so you **must restart the runtime
afterward** — see the next cell.

In [ ]:
!pip install -q "colpali-engine==0.3.17" "transformers>=5.3,<6" "torch>=2.2,<2.12" "peft>=0.18,<0.20" "qdrant-client>=1.17,<2.0" pypdfium2 pillow

## ⚠ RESTART THE RUNTIME after the install above — then skip back to here

The install swaps Colab's preinstalled `transformers` / `torch` / `peft` for the versions
ColQwen2.5 needs. Colab keeps the **old** versions loaded in the running kernel until you
restart — running the cells below without restarting is the usual cause of *"every cell errors,
reconnect, retry"*.

**Runtime → Restart session**, then run from the **next cell down** (cell 2 / clone). Do **NOT**
re-run the install cell after restarting.

import os, sys, subprocess

# RECOMMENDED: make this repo PUBLIC — then no token is needed and nothing can leak.
# (The repo is source only; PDFs and compliance.db are gitignored.)
# Private alternative: set a Colab secret GH_TOKEN (fine-grained PAT, Contents:Read on THIS repo).
REPO_URL = os.environ.get("REPO_URL", "https://github.com/aatif101/pfizer-externship.git")
REPO_DIR = "/content/pfizer-externship"

token = None
try:
    from google.colab import userdata  # type: ignore
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None  # public repo, or src/ uploaded under /content

if REPO_URL and not os.path.isdir(REPO_DIR):
    clone_url = REPO_URL
    if token and clone_url.startswith("https://github.com/"):
        clone_url = clone_url.replace("https://github.com/", f"https://{token}@github.com/")
    # capture_output so a failure NEVER echoes the token into the traceback.
    res = subprocess.run(
        ["git", "clone", "--depth", "1", clone_url, REPO_DIR],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        stderr = (res.stderr or "").replace(token or "\0", "***")  # redact token if present
        raise RuntimeError(
            f"git clone failed (exit {res.returncode}).\n"
            "Fix: make the repo PUBLIC (simplest), or ensure the GH_TOKEN secret is a "
            "fine-grained PAT with Contents:Read granted to THIS repo.\n"
            f"git said:\n{stderr}"
        )

if os.path.isdir(REPO_DIR):
    sys.path.insert(0, REPO_DIR)
elif os.path.isdir("/content/src"):
    sys.path.insert(0, "/content")  # fallback: src/ uploaded directly under /content

import src.retrieval.visual.collection  # noqa: F401  fail fast if the path is wrong
print("src.retrieval.visual importable:", True)

In [ ]:
## 3. Version print + smoke embed (resolves RESEARCH Open Questions A1–A3)

Print the exact `(colpali-engine, transformers, torch)` triple that loaded, then load the model
via the shared `embedder.load_colqwen` seam and embed ONE image to confirm the per-token
embedding dim (expect **128**, RESEARCH A1) BEFORE the full build. **Record the printed version
triple + dim into 05-04-SUMMARY.md.** colpali-engine 0.3.17 requires transformers>=5.3 (now pinned);
if the model load still throws, paste the traceback and pin the exact loadable pair here.

import gc
import importlib.metadata as _md
import colpali_engine  # noqa: F401  (import proves it loaded; version comes from metadata)
import transformers
import torch
from PIL import Image


def _ver(pkg: str) -> str:
    # colpali_engine has no __version__ attribute — read the installed dist version.
    try:
        return _md.version(pkg)
    except Exception:
        return "unknown"


print("colpali-engine:", _ver("colpali-engine"))
print("transformers   :", _ver("transformers"))
print("torch          :", _ver("torch"))
print("cuda available :", torch.cuda.is_available())
assert torch.cuda.is_available(), "This notebook requires a GPU (Colab L4). Runtime -> Change runtime type -> L4 GPU."
print("gpu            :", torch.cuda.get_device_name(0))

from src.retrieval.visual.embedder import load_colqwen, embed_images, embed_queries, pooled_vectors_for_image, process_image_batch

model, processor = load_colqwen()  # vidore/colqwen2.5-v0.2, bf16, cuda:0

# RESEARCH A1/A2: confirm dim==128 and inspect the dynamic-grid attributes BEFORE the full build.
smoke_img = Image.new("RGB", (768, 1024), color="white")
smoke_emb = embed_images(model, processor, [smoke_img])
print("smoke image embedding shape:", tuple(smoke_emb.shape), "-> per-token dim =", int(smoke_emb.shape[-1]))
assert smoke_emb.shape[-1] == 128, f"expected per-token dim 128, got {smoke_emb.shape[-1]} (update collection size if this fires)"
print("model.patch_size =", getattr(model, "patch_size", None), " spatial_merge_size =", getattr(model, "spatial_merge_size", None))
del smoke_emb; torch.cuda.empty_cache(); gc.collect()

In [ ]:
import gc
import colpali_engine
import transformers
import torch
from PIL import Image

print("colpali-engine:", colpali_engine.__version__)
print("transformers   :", transformers.__version__)
print("torch          :", torch.__version__)
print("cuda available :", torch.cuda.is_available())
assert torch.cuda.is_available(), "This notebook requires a GPU (Colab L4). Runtime -> Change runtime type -> L4 GPU."
print("gpu            :", torch.cuda.get_device_name(0))

from src.retrieval.visual.embedder import load_colqwen, embed_images, embed_queries, pooled_vectors_for_image, process_image_batch

model, processor = load_colqwen()  # vidore/colqwen2.5-v0.2, bf16, cuda:0

# RESEARCH A1/A2: confirm dim==128 and inspect the dynamic-grid attributes BEFORE the full build.
smoke_img = Image.new("RGB", (768, 1024), color="white")
smoke_emb = embed_images(model, processor, [smoke_img])
print("smoke image embedding shape:", tuple(smoke_emb.shape), "-> per-token dim =", int(smoke_emb.shape[-1]))
assert smoke_emb.shape[-1] == 128, f"expected per-token dim 128, got {smoke_emb.shape[-1]} (update collection size if this fires)"
print("model.patch_size =", getattr(model, "patch_size", None), " spatial_merge_size =", getattr(model, "spatial_merge_size", None))
del smoke_emb; torch.cuda.empty_cache(); gc.collect()

## 4. Upload `compliance.db` and read pages from `pages.image_blob`

`compliance.db` carries the per-page `image_blob` (the reproducible 150-DPI PNG). We rasterize
from the stored blob via the shared `pooling.blob_to_image` — **never** from the gitignored
source PDFs. Only page identities + image bytes leave the DB; no full page text is dumped to output.

In [ ]:
import sqlite3

DB_PATH = "/content/compliance.db"
if not os.path.exists(DB_PATH):
    from google.colab import files  # type: ignore
    print("Upload compliance.db (with pages.image_blob) ...")
    uploaded = files.upload()
    name = next(iter(uploaded))
    if name != "compliance.db":
        os.replace(name, DB_PATH)
assert os.path.exists(DB_PATH), "compliance.db not found at /content/compliance.db"

from src.retrieval.visual.pooling import blob_to_image

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
# Parameterized-shape SQL (no string interpolation): every page incl. image-only ones.
rows = conn.execute(
    "SELECT p.doc_id AS doc_id, p.page_num AS page_num, d.filename AS filename, p.image_blob AS image_blob "
    "FROM pages p JOIN documents d ON d.doc_id = p.doc_id "
    "WHERE p.image_blob IS NOT NULL "
    "ORDER BY p.doc_id, p.page_num"
).fetchall()
conn.close()

pages = [
    {
        "doc_id": r["doc_id"],
        "page_num": int(r["page_num"]),  # 0-indexed throughout (RESEARCH Pitfall 6)
        "filename": r["filename"],
        "image": blob_to_image(bytes(r["image_blob"])),
    }
    for r in rows
]
print(f"pages to embed: {len(pages)}  across {len({p['doc_id'] for p in pages})} documents")

## 5. Build the `sdf_page_images` Qdrant collection (3 named vectors)

Uses the shared `collection.build_vectors_config()` (original HNSW-off + mean_pooling_rows/
columns) and `collection.build_upsert_point(...)`. Embedding runs in bf16 at `batch_size=2`
with `empty_cache()+gc.collect()` between batches (RESEARCH Pitfall 3). For each page we derive
the row/column pooled multivectors via the shared `embedder.pooled_vectors_for_image`
(which delegates the reshape to the offline-tested `pooling.mean_pool_rows_cols`). Payload
carries 0-indexed `page_num` + `doc_id` ONLY — no image bytes, no page text.

In [ ]:
from qdrant_client import QdrantClient
from src.retrieval.visual.collection import build_vectors_config, collection_name, build_upsert_point

VERSION = 1
BATCH_SIZE = 2
QDRANT_PATH = "/content/qdrant_storage"

client = QdrantClient(path=QDRANT_PATH)
name = collection_name(VERSION)
if client.collection_exists(name):
    client.delete_collection(name)
client.create_collection(collection_name=name, vectors_config=build_vectors_config())

def _to_list(t):
    return t.detach().to(torch.float32).cpu().numpy().tolist()

upserted = 0
for start in range(0, len(pages), BATCH_SIZE):
    batch = pages[start : start + BATCH_SIZE]
    images = [p["image"] for p in batch]
    batch_images = process_image_batch(processor, images).to(model.device)
    with torch.no_grad():
        image_embeddings = model(**batch_images)  # [B, seq, 128]
    points = []
    for i, p in enumerate(batch):
        pooled_rows, pooled_cols = pooled_vectors_for_image(
            model, processor, batch_images, image_embeddings, i, p["image"].size
        )
        points.append(
            build_upsert_point(
                p["doc_id"], p["page_num"],
                _to_list(image_embeddings[i]),  # original full multivector
                _to_list(pooled_rows),
                _to_list(pooled_cols),
            )
        )
    client.upsert(collection_name=name, points=points)
    upserted += len(points)
    del image_embeddings, batch_images
    torch.cuda.empty_cache(); gc.collect()
    print(f"  upserted {upserted}/{len(pages)} pages", end="\r")

info = client.get_collection(name)
print(f"\ncollection {name}: points={client.count(name).count} indexed_vectors_count={info.indexed_vectors_count}")

# Persist a versioned visual run record (counts/run_id/model_version only).
from src.retrieval.visual.run import build_visual_index_run
pages_meta = [(p["doc_id"], p["page_num"], p["filename"]) for p in pages]
visual_run = build_visual_index_run(DB_PATH, pages_meta, model_version="vidore/colqwen2.5-v0.2", version=VERSION)
print("visual run_id:", visual_run.run_id, "| indexed pages:", visual_run.indexed_page_count)

## 6. Two-stage visual retrieval over the gold queries

Embed each (repaired) gold query, run the canonical two-stage query via the shared
`querier.build_query_payload` (two mean-pooled HNSW prefetches -> `using="original"` MAX_SIM
rerank), and map the response with `querier.map_response_to_candidates` (0-indexed identity).

In [ ]:
from src.retrieval.visual.querier import build_query_payload, map_response_to_candidates

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
gold_queries = [
    {"query_id": r["query_id"], "query_text": r["query_text"]}
    for r in conn.execute("SELECT query_id, query_text FROM gold_retrieval_queries ORDER BY query_id").fetchall()
]
gold_targets = {}
for q in gold_queries:
    tgt = conn.execute(
        "SELECT doc_id, page_num FROM gold_retrieval_targets WHERE query_id = ? ORDER BY doc_id, page_num",
        (q["query_id"],),
    ).fetchall()
    gold_targets[q["query_id"]] = [(str(t["doc_id"]), int(t["page_num"])) for t in tgt]
conn.close()
print(f"gold queries: {len(gold_queries)}")

TOP_K = 10
visual_ranked_by_query = {}
for q in gold_queries:
    q_emb = embed_queries(model, processor, [q["query_text"]])
    payload = build_query_payload(_to_list(q_emb[0]), prefetch_limit=200, search_limit=TOP_K, version=VERSION)
    response = client.query_points(**payload)
    candidates = map_response_to_candidates(response.points)
    visual_ranked_by_query[q["query_id"]] = [(c.doc_id, c.page_num) for c in candidates]
    del q_emb; torch.cuda.empty_cache(); gc.collect()
print("two-stage retrieval complete for all gold queries")

## 7. Evaluate text-only vs visual-fused on the SAME gold set + print the metrics table

Both modes use the UNCHANGED metric functions (`compute_retrieval_recall_at_k`,
`compute_page_level_citation_accuracy`). Text-only uses the existing `retrieve_evidence`
page identities; visual-fused fuses the visual ranking with the text ranking via the shared
`fusion.rrf_fuse` (RRF k=60). Metrics key only on `(doc_id, page_num)` — no change required.

In [ ]:
from src.retrieval.retriever import retrieve_evidence
from src.retrieval.visual.fusion import rrf_fuse
from src.eval.retrieval_metrics import compute_retrieval_recall_at_k, compute_page_level_citation_accuracy

K_VALUES = (5, 10)

# Text-only ranked page identities (existing path, unchanged).
text_ranked_by_query = {}
for q in gold_queries:
    res = retrieve_evidence(DB_PATH, q["query_text"], top_k=max(K_VALUES))
    text_ranked_by_query[q["query_id"]] = [(h.doc_id, int(h.page_num)) for h in res.hits]

# Visual-fused ranked page identities = RRF(visual, text).
fused_ranked_by_query = {}
for q in gold_queries:
    qid = q["query_id"]
    fused = rrf_fuse(visual_ranked_by_query.get(qid, []), text_ranked_by_query.get(qid, []), k=60)
    fused_ranked_by_query[qid] = [page_key for page_key, _score in fused]

def _eval_mode(ranked_by_query):
    out = {}
    for k in K_VALUES:
        retrieved = {qid: [(d, p, 1.0) for d, p in ranked[:k]] for qid, ranked in ranked_by_query.items()}
        recall = compute_retrieval_recall_at_k(gold_targets, retrieved, k=k)
        cited = {qid: ranked[:k] for qid, ranked in ranked_by_query.items()}
        citation = compute_page_level_citation_accuracy(gold_targets, cited)
        out[f"recall@{k}"] = recall.macro_recall
        out[f"citation_acc@{k}"] = float(citation["macro_accuracy"])
    return out

text_metrics = _eval_mode(text_ranked_by_query)
fused_metrics = _eval_mode(fused_ranked_by_query)

metric_keys = [f"recall@{k}" for k in K_VALUES] + [f"citation_acc@{k}" for k in K_VALUES]
print(f"{'metric':<18}{'text-only':>12}{'visual-fused':>14}{'delta':>10}")
print("-" * 54)
for key in metric_keys:
    t = text_metrics[key]; f = fused_metrics[key]
    print(f"{key:<18}{t:>12.3f}{f:>14.3f}{(f - t):>+10.3f}")
print("\n>>> Record this table into 05-04-SUMMARY.md (real VISUAL-01/VISUAL-02 numbers).")

## 8. Example-3 proof: the four `rq_ex3_*` image-only gold pages appear in visual top-k

Doc `5543408c4dacc48b`, gold page **2 (0-indexed)** is scanned image-only — text recall@5 is
structurally 0 on all four `rq_ex3_*` queries (the text indexer excludes empty-text pages).
This cell prints, for each `rq_ex3_*` query, whether each gold page appears in the **visual**
and **fused** top-k — the core proof that the visual tier makes image-only pages retrievable.

In [ ]:
rq_ex3_ids = [q["query_id"] for q in gold_queries if str(q["query_id"]).startswith("rq_ex3")]
print(f"rq_ex3 queries found: {rq_ex3_ids}\n")

PROOF_K = 5
for qid in rq_ex3_ids:
    targets = set(gold_targets.get(qid, []))
    visual_topk = set(visual_ranked_by_query.get(qid, [])[:PROOF_K])
    fused_topk = set(fused_ranked_by_query.get(qid, [])[:PROOF_K])
    text_topk = set(text_ranked_by_query.get(qid, [])[:PROOF_K])
    print(f"{qid}: gold={sorted(targets)}")
    for tgt in sorted(targets):
        print(
            f"    page {tgt}: text@{PROOF_K}={'HIT' if tgt in text_topk else 'miss':<4} "
            f"visual@{PROOF_K}={'HIT' if tgt in visual_topk else 'miss':<4} "
            f"fused@{PROOF_K}={'HIT' if tgt in fused_topk else 'miss'}"
        )
print("\n>>> Expectation: rq_ex3 gold pages are MISS for text but HIT for visual/fused — recall lift above the 0.647 text-only ceiling.")

import json, shutil

manifest = {
    "collection_name": name,
    "run_id": visual_run.run_id,
    "point_count": int(client.count(name).count),
    "model_version": "vidore/colqwen2.5-v0.2",
    "colpali_engine": _ver("colpali-engine"),  # _ver defined in the smoke cell (§3)
    "transformers": _ver("transformers"),
    "torch": _ver("torch"),
}
with open("/content/visual_index_manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
shutil.make_archive("/content/qdrant_storage_artifact", "zip", QDRANT_PATH)
print("manifest:", json.dumps(manifest, indent=2))
print("artifact: /content/qdrant_storage_artifact.zip (download for the deferred demo; do NOT commit)")

In [ ]:
import json, shutil

manifest = {
    "collection_name": name,
    "run_id": visual_run.run_id,
    "point_count": int(client.count(name).count),
    "model_version": "vidore/colqwen2.5-v0.2",
    "colpali_engine": colpali_engine.__version__,
    "transformers": transformers.__version__,
    "torch": torch.__version__,
}
with open("/content/visual_index_manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
shutil.make_archive("/content/qdrant_storage_artifact", "zip", QDRANT_PATH)
print("manifest:", json.dumps(manifest, indent=2))
print("artifact: /content/qdrant_storage_artifact.zip (download for the deferred demo; do NOT commit)")